In [ ]:
import os, warnings, torch

import numpy as np
import scanpy as sc
import pandas as pd

from datasets.data_manager_STARmap_PLUS import AD_Mouse
from utils.notebook_graph_utils import (
    build_graph_model,
    cell_type_gradients,
    feature_gradients,
    load_graph_checkpoint,
    predict_single_graph,
)
from utils.utils import *
from utils.utils_dataloader import *
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


In [ ]:
%run ./args/args_STARmap_PLUS.py
args = args

### Load dataset

In [ ]:
# create the dataset
dataset = AD_Mouse(
    AD_adata_path=args.AD_adata_path,
    Wild_type_adata_path=args.Wild_type_adata_path,
    n_top_genes=args.n_top_genes,
    graph_k=args.graph_k,
    val_ratio=args.val_ratio,
    mask_seed=args.seed,
)


### Load model

In [ ]:
# create the model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_graph_model(args, dataset, use_cell_type=args.use_cell_type).to(device)
model = load_graph_checkpoint(model, 'WholeSliceGraphTransformer_ct_STARmap_PLUS.pth', device=device)


### Inference 

In [ ]:
graph = dataset.testing[0]
pred_value, _, test_mask = predict_single_graph(model, graph, device=device, apply_sigmoid=True)
selected_mask = test_mask & (pred_value[:, 0] > 0.05)
print(f"Selected {int(selected_mask.sum())} nodes for gene attribution.")


### Calculate and Visualize the gradients

In [ ]:
gradients, outputs, attribution_mask = feature_gradients(
    model,
    graph,
    target_index=0,
    split='test',
    device=device,
    apply_sigmoid=True,
    selection_mask=selected_mask,
)

attributions = gradients.numpy()
attributions = np.abs(attributions)
grad_norm = attributions.sum(axis=0)


In [ ]:
num_genes = 30

top_indices = np.argsort(grad_norm)[-num_genes:]
temp_mask = grad_norm >= grad_norm[top_indices[0]]

highly_correlated_genes_0 = dataset.source_panel[top_indices]

In [ ]:
sorted_data = np.sort(grad_norm.squeeze())[::-1]
indices = np.arange(len(sorted_data))

fig, ax = plt.subplots(1, figsize=(6.5, 2), dpi=300)

plt.scatter(indices[0:30], sorted_data[0:30], s=2, color='#6F1D57')
plt.scatter(29 + (indices[30:] -29)/40, sorted_data[30:], s=0.2, color='#FCBD8B')

num_genes
for i in range(num_genes):
    plt.annotate(highly_correlated_genes_0[num_genes-i -1],
                 xy=(i, sorted_data[i]),
                 xytext=(0, 5),  
                 textcoords='offset points',
                 ha='center',  
                 rotation=90,
                 fontsize=4.5)  

                 
plt.fill_between(indices[0:30], sorted_data[0:30], color='#E8789A', alpha=0.4)
plt.fill_between(29 + (indices[30:]- 29 )/40, sorted_data[30:], color='#EDDAB9', alpha=0.4)

custom_xticks = [0, 30, 60]
custom_xticklabels = ['0', '30', '1230']

ax.set_xticks(custom_xticks)
ax.set_xticklabels(custom_xticklabels, fontsize=5)
plt.yticks(fontsize=5)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.show()